In [ ]:
SIMILARITY_THRESHOLD = 0.7

In [1]:
import sys
print(sys.executable)

from praatio import textgrid

tg = textgrid.openTextgrid("../output/twinkle.TextGrid", includeEmptyIntervals=False)

word_tier = tg.getTier("words")
phone_tier = tg.getTier("phones")

for word in word_tier.entries:
    print(word)

C:\Users\georg\anaconda3\envs\mfa_env\python.exe
Interval(start=1.89, end=3.12, label='twinkle')
Interval(start=3.12, end=4.23, label='twinkle')
Interval(start=4.23, end=5.05, label='little')
Interval(start=5.05, end=5.85, label='star')
Interval(start=6.18, end=6.64, label='how')
Interval(start=6.64, end=6.67, label='i')
Interval(start=6.67, end=7.1, label='wonder')
Interval(start=7.1, end=8.66, label='what')
Interval(start=8.85, end=9.48, label='you')
Interval(start=9.63, end=10.22, label='are')
Interval(start=10.94, end=11.67, label='sky')
Interval(start=12.29, end=12.94, label='high')
Interval(start=13.6, end=14.51, label='skillet')
Interval(start=15.14, end=15.81, label='skill')
Interval(start=16.15, end=16.6, label='at')


In [2]:
for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    phonemes = []
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    print(word_label, "→", phonemes)

twinkle → ['T', 'W', 'IH1', 'NG', 'K', 'AH0', 'L']
twinkle → ['T', 'W', 'IH1', 'NG', 'K', 'AH0', 'L']
little → ['L', 'IH1', 'T', 'AH0', 'L']
star → ['S', 'T', 'AA1', 'R']
how → ['HH', 'AW1']
i → ['AY1']
wonder → ['W', 'AH1', 'N', 'D', 'ER0']
what → ['HH', 'W', 'AH1', 'T']
you → ['Y', 'UW1']
are → ['AA1', 'R']
sky → ['S', 'K', 'AY1']
high → ['HH', 'AY1']
skillet → ['S', 'K', 'IH1', 'L', 'AH0', 'T']
skill → ['S', 'K', 'IH1', 'L']
at → ['AE1', 'T']


In [3]:
word_to_phonemes = {}

for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    
    phonemes = []
    
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    
    word_label = word_label.lower()
    
    if word_label not in word_to_phonemes:
        word_to_phonemes[word_label] = phonemes

print(word_to_phonemes)

{'twinkle': ['T', 'W', 'IH1', 'NG', 'K', 'AH0', 'L'], 'little': ['L', 'IH1', 'T', 'AH0', 'L'], 'star': ['S', 'T', 'AA1', 'R'], 'how': ['HH', 'AW1'], 'i': ['AY1'], 'wonder': ['W', 'AH1', 'N', 'D', 'ER0'], 'what': ['HH', 'W', 'AH1', 'T'], 'you': ['Y', 'UW1'], 'are': ['AA1', 'R'], 'sky': ['S', 'K', 'AY1'], 'high': ['HH', 'AY1'], 'skillet': ['S', 'K', 'IH1', 'L', 'AH0', 'T'], 'skill': ['S', 'K', 'IH1', 'L'], 'at': ['AE1', 'T']}


In [4]:
def extract_rhyme_unit(phonemes, debug=False):

    stressed_index = None

    for i in range(len(phonemes) - 1, -1, -1):
        if '1' in phonemes[i]:
            stressed_index = i
            break

    if stressed_index is None:
        for i in range(len(phonemes) - 1, -1, -1):
            if '2' in phonemes[i] or '0' in phonemes[i]:
                stressed_index = i
                break

    if stressed_index is None:
        stressed_index = 0

    rhyme_unit = phonemes[stressed_index:]

    if debug:
        print("PHONEMES:", phonemes)
        print("STRESS INDEX:", stressed_index)
        print("RHYME UNIT:", rhyme_unit)

    return rhyme_unit

In [5]:
def extract_end_words(lyrics_path, word_to_phonemes):

    result = []

    with open(lyrics_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):

        line = line.strip()

        if not line:
            continue

        words = line.lower().split()
        end_word = words[-1]

        if end_word not in word_to_phonemes:
            print(f"WARNING: phonemes not found for '{end_word}'")
            continue

        phonemes = word_to_phonemes[end_word]
        rhyme_unit = extract_rhyme_unit(phonemes)

        result.append({
            "line_index": i,
            "line_text": line,
            "end_word": end_word,
            "phonemes": phonemes,
            "rhyme_unit": rhyme_unit
        })

    return result

In [6]:
def rhyme_similarity(rhymeA, rhymeB):

    # ----- VOWEL MATCH -----
    vowelA = rhymeA[0]
    vowelB = rhymeB[0]

    vowel_score = 1.0 if vowelA == vowelB else 0.0


    # ----- CODA OVERLAP -----
    codaA = rhymeA[1:]
    codaB = rhymeB[1:]

    if len(codaA) == 0 and len(codaB) == 0:
        coda_score = 1.0
    else:
        overlap = len(set(codaA) & set(codaB))
        max_len = max(len(codaA), len(codaB))

        if max_len == 0:
            coda_score = 0
        else:
            coda_score = overlap / max_len


    # ----- LENGTH SIMILARITY -----
    lenA = len(rhymeA)
    lenB = len(rhymeB)

    length_score = 1 - abs(lenA - lenB) / max(lenA, lenB)


    # ----- WEIGHTED FINAL SCORE -----
    similarity = (
        0.6 * vowel_score +
        0.3 * coda_score +
        0.1 * length_score
    )

    return similarity

In [7]:
import itertools

def compute_similarity_pairs(end_word_objects):

    similarity_results = []

    for a, b in itertools.combinations(end_word_objects, 2):

        wordA = a["end_word"]
        wordB = b["end_word"]

        rhymeA = a["rhyme_unit"]
        rhymeB = b["rhyme_unit"]

        score = rhyme_similarity(rhymeA, rhymeB)

        similarity_results.append({
            "wordA": wordA,
            "wordB": wordB,
            "similarity": score
        })

    return similarity_results

In [8]:
rhyme_candidates = extract_end_words("../input/twinkle.txt", word_to_phonemes)

for r in rhyme_candidates:
    print(r)

{'line_index': 0, 'line_text': 'Twinkle twinkle little star', 'end_word': 'star', 'phonemes': ['S', 'T', 'AA1', 'R'], 'rhyme_unit': ['AA1', 'R']}
{'line_index': 1, 'line_text': 'How I wonder what you are', 'end_word': 'are', 'phonemes': ['AA1', 'R'], 'rhyme_unit': ['AA1', 'R']}
{'line_index': 2, 'line_text': 'sky', 'end_word': 'sky', 'phonemes': ['S', 'K', 'AY1'], 'rhyme_unit': ['AY1']}
{'line_index': 3, 'line_text': 'high', 'end_word': 'high', 'phonemes': ['HH', 'AY1'], 'rhyme_unit': ['AY1']}
{'line_index': 4, 'line_text': 'skillet', 'end_word': 'skillet', 'phonemes': ['S', 'K', 'IH1', 'L', 'AH0', 'T'], 'rhyme_unit': ['IH1', 'L', 'AH0', 'T']}
{'line_index': 5, 'line_text': 'skill at', 'end_word': 'at', 'phonemes': ['AE1', 'T'], 'rhyme_unit': ['AE1', 'T']}


In [9]:
similarity_pairs = compute_similarity_pairs(rhyme_candidates)

for pair in similarity_pairs:
    print(pair)

{'wordA': 'star', 'wordB': 'are', 'similarity': 0.9999999999999999}
{'wordA': 'star', 'wordB': 'sky', 'similarity': 0.05}
{'wordA': 'star', 'wordB': 'high', 'similarity': 0.05}
{'wordA': 'star', 'wordB': 'skillet', 'similarity': 0.05}
{'wordA': 'star', 'wordB': 'at', 'similarity': 0.1}
{'wordA': 'are', 'wordB': 'sky', 'similarity': 0.05}
{'wordA': 'are', 'wordB': 'high', 'similarity': 0.05}
{'wordA': 'are', 'wordB': 'skillet', 'similarity': 0.05}
{'wordA': 'are', 'wordB': 'at', 'similarity': 0.1}
{'wordA': 'sky', 'wordB': 'high', 'similarity': 0.9999999999999999}
{'wordA': 'sky', 'wordB': 'skillet', 'similarity': 0.025}
{'wordA': 'sky', 'wordB': 'at', 'similarity': 0.05}
{'wordA': 'high', 'wordB': 'skillet', 'similarity': 0.025}
{'wordA': 'high', 'wordB': 'at', 'similarity': 0.05}
{'wordA': 'skillet', 'wordB': 'at', 'similarity': 0.15}


In [10]:
def build_similarity_matrix(rhyme_candidates, similarity_pairs):

    # collect all words
    words = [obj["end_word"] for obj in rhyme_candidates]

    # initialize matrix
    matrix = {w: {v: 0.0 for v in words} for w in words}

    # diagonal = identical words
    for w in words:
        matrix[w][w] = 1.0

    # fill matrix using computed pairs
    for pair in similarity_pairs:

        wA = pair["wordA"]
        wB = pair["wordB"]
        score = pair["similarity"]

        matrix[wA][wB] = score
        matrix[wB][wA] = score

    return matrix

In [11]:
similarity_matrix = build_similarity_matrix(rhyme_candidates, similarity_pairs)

for w1 in similarity_matrix:
    print(w1, similarity_matrix[w1])

star {'star': 1.0, 'are': 0.9999999999999999, 'sky': 0.05, 'high': 0.05, 'skillet': 0.05, 'at': 0.1}
are {'star': 0.9999999999999999, 'are': 1.0, 'sky': 0.05, 'high': 0.05, 'skillet': 0.05, 'at': 0.1}
sky {'star': 0.05, 'are': 0.05, 'sky': 1.0, 'high': 0.9999999999999999, 'skillet': 0.025, 'at': 0.05}
high {'star': 0.05, 'are': 0.05, 'sky': 0.9999999999999999, 'high': 1.0, 'skillet': 0.025, 'at': 0.05}
skillet {'star': 0.05, 'are': 0.05, 'sky': 0.025, 'high': 0.025, 'skillet': 1.0, 'at': 0.15}
at {'star': 0.1, 'are': 0.1, 'sky': 0.05, 'high': 0.05, 'skillet': 0.15, 'at': 1.0}


In [ ]:
def cluster_rhymes(similarity_matrix, threshold=SIMILARITY_THRESHOLD):

    words = list(similarity_matrix.keys())
    clusters = {}
    cluster_labels = {}
    current_cluster = "A"

    for word in words:

        if word in cluster_labels:
            continue

        clusters[current_cluster] = [word]
        cluster_labels[word] = current_cluster

        for other in words:

            if other == word:
                continue

            score = similarity_matrix[word][other]

            if score >= threshold and other not in cluster_labels:
                clusters[current_cluster].append(other)
                cluster_labels[other] = current_cluster

        current_cluster = chr(ord(current_cluster) + 1)

    return cluster_labels, clusters

In [ ]:
cluster_labels, clusters = cluster_rhymes(similarity_matrix, threshold=SIMILARITY_THRESHOLD)

print("Clusters:")
print(clusters)

print("\nWord → Cluster Mapping:")
print(cluster_labels)

In [12]:
expected_clusters = {
    "star": "A",
    "are": "A",
    "sky": "B",
    "high": "B",
    "skillet": "C",
    "at": "D"
}

In [13]:
for r in rhyme_candidates:
    word = r["end_word"]
    r["expected_cluster"] = expected_clusters.get(word, None)

for r in rhyme_candidates:
    print(r)

{'line_index': 0, 'line_text': 'Twinkle twinkle little star', 'end_word': 'star', 'phonemes': ['S', 'T', 'AA1', 'R'], 'rhyme_unit': ['AA1', 'R'], 'expected_cluster': 'A'}
{'line_index': 1, 'line_text': 'How I wonder what you are', 'end_word': 'are', 'phonemes': ['AA1', 'R'], 'rhyme_unit': ['AA1', 'R'], 'expected_cluster': 'A'}
{'line_index': 2, 'line_text': 'sky', 'end_word': 'sky', 'phonemes': ['S', 'K', 'AY1'], 'rhyme_unit': ['AY1'], 'expected_cluster': 'B'}
{'line_index': 3, 'line_text': 'high', 'end_word': 'high', 'phonemes': ['HH', 'AY1'], 'rhyme_unit': ['AY1'], 'expected_cluster': 'B'}
{'line_index': 4, 'line_text': 'skillet', 'end_word': 'skillet', 'phonemes': ['S', 'K', 'IH1', 'L', 'AH0', 'T'], 'rhyme_unit': ['IH1', 'L', 'AH0', 'T'], 'expected_cluster': 'C'}
{'line_index': 5, 'line_text': 'skill at', 'end_word': 'at', 'phonemes': ['AE1', 'T'], 'rhyme_unit': ['AE1', 'T'], 'expected_cluster': 'D'}


In [14]:
def cluster_exact_rhymes(rhyme_candidates):

    clusters = {}
    cluster_labels = {}
    current_label = 0

    for entry in rhyme_candidates:

        rhyme_tuple = tuple(entry["rhyme_unit"])

        if rhyme_tuple not in clusters:

            label = chr(ord('A') + current_label)
            clusters[rhyme_tuple] = label
            current_label += 1

        cluster_labels[entry["end_word"]] = clusters[rhyme_tuple]

    return cluster_labels

In [15]:
def debug_rhyme_analysis(rhyme_candidates, predicted_clusters):

    print("\n===== RHYME DEBUG REPORT =====\n")

    for r in rhyme_candidates:

        word = r["end_word"]
        phonemes = r["phonemes"]
        rhyme_unit = r["rhyme_unit"]
        expected = r.get("expected_cluster", None)
        predicted = predicted_clusters.get(word, None)

        print(f"WORD: {word}")
        print(f"PHONEMES: {phonemes}")
        print(f"RHYME UNIT: {rhyme_unit}")
        print(f"EXPECTED CLUSTER: {expected}")
        print(f"PREDICTED CLUSTER: {predicted}")

        if expected == predicted:
            print("STATUS: ✓ CORRECT")
        else:
            print("STATUS: ✗ MISMATCH")

        print("-" * 40)

In [16]:
def evaluate_rhyme_detection(rhyme_candidates, predicted_clusters):

    total = 0
    correct = 0
    mismatches = []

    for r in rhyme_candidates:

        word = r["end_word"]
        expected = r.get("expected_cluster")
        predicted = predicted_clusters.get(word)

        if expected is None:
            continue

        total += 1

        if expected == predicted:
            correct += 1
        else:
            mismatches.append({
                "word": word,
                "expected": expected,
                "predicted": predicted
            })

    accuracy = correct / total if total > 0 else 0

    print("\n===== RHYME EVALUATION =====\n")
    print(f"TOTAL WORDS: {total}")
    print(f"CORRECT: {correct}")
    print(f"ACCURACY: {accuracy:.2f}")

    if mismatches:
        print("\nMISMATCHES:")
        for m in mismatches:
            print(m)

    return accuracy, mismatches

In [17]:
clusters = cluster_exact_rhymes(rhyme_candidates)

print(clusters)

{'star': 'A', 'are': 'A', 'sky': 'B', 'high': 'B', 'skillet': 'C', 'at': 'D'}


In [18]:
clusters = cluster_exact_rhymes(rhyme_candidates)

debug_rhyme_analysis(rhyme_candidates, clusters)


===== RHYME DEBUG REPORT =====

WORD: star
PHONEMES: ['S', 'T', 'AA1', 'R']
RHYME UNIT: ['AA1', 'R']
EXPECTED CLUSTER: A
PREDICTED CLUSTER: A
STATUS: ✓ CORRECT
----------------------------------------
WORD: are
PHONEMES: ['AA1', 'R']
RHYME UNIT: ['AA1', 'R']
EXPECTED CLUSTER: A
PREDICTED CLUSTER: A
STATUS: ✓ CORRECT
----------------------------------------
WORD: sky
PHONEMES: ['S', 'K', 'AY1']
RHYME UNIT: ['AY1']
EXPECTED CLUSTER: B
PREDICTED CLUSTER: B
STATUS: ✓ CORRECT
----------------------------------------
WORD: high
PHONEMES: ['HH', 'AY1']
RHYME UNIT: ['AY1']
EXPECTED CLUSTER: B
PREDICTED CLUSTER: B
STATUS: ✓ CORRECT
----------------------------------------
WORD: skillet
PHONEMES: ['S', 'K', 'IH1', 'L', 'AH0', 'T']
RHYME UNIT: ['IH1', 'L', 'AH0', 'T']
EXPECTED CLUSTER: C
PREDICTED CLUSTER: C
STATUS: ✓ CORRECT
----------------------------------------
WORD: at
PHONEMES: ['AE1', 'T']
RHYME UNIT: ['AE1', 'T']
EXPECTED CLUSTER: D
PREDICTED CLUSTER: D
STATUS: ✓ CORRECT
---------------

In [19]:
accuracy, mismatches = evaluate_rhyme_detection(rhyme_candidates, clusters)


===== RHYME EVALUATION =====

TOTAL WORDS: 6
CORRECT: 6
ACCURACY: 1.00


In [20]:
def generate_rhyme_html(rhyme_candidates, clusters, output_file="../generated_html/rhyme_visualization.html"):

    colors = [
        "#ffcccc", "#cce5ff", "#ccffcc",
        "#ffe6cc", "#ffffcc", "#e6ccff",
        "#ffd6e7", "#ccfff5", "#e6f2ff"
    ]

    # assign a color to each cluster label
    cluster_labels = sorted(set(clusters.values()))
    cluster_colors = {}

    for i, label in enumerate(cluster_labels):
        cluster_colors[label] = colors[i % len(colors)]

    # generate CSS classes
    css_rules = []
    for label, color in cluster_colors.items():
        css_rules.append(f".rhyme-{label} {{ background-color: {color}; padding:2px; }}")

    css_block = "\n".join(css_rules)

    html_lines = []

    for entry in rhyme_candidates:

        line = entry["line_text"]
        end_word = entry["end_word"]
        label = clusters[end_word]

        highlighted_word = f'<span class="rhyme-{label}">{end_word}</span>'

        highlighted_line = line.rsplit(end_word, 1)[0] + highlighted_word

        html_lines.append(highlighted_line)

    html_content = "<br>\n".join(html_lines)

    full_html = f"""
    <html>
    <head>
    <title>Verse DNA Rhyme Visualization</title>
    <style>
    body {{ font-family: Arial; font-size: 18px; }}
    {css_block}
    </style>
    </head>
    <body>
    {html_content}
    </body>
    </html>
    """

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(full_html)

    print("HTML file generated:", output_file)

In [21]:
clusters = cluster_exact_rhymes(rhyme_candidates)

generate_rhyme_html(rhyme_candidates, clusters)

HTML file generated: generated_html/rhyme_visualization.html
